In [ ]:

# azureml-core of version 1.0.72 or higher is required
# azureml-dataprep[pandas] of version 1.1.34 or higher is required
from azureml.core import Workspace, Dataset

subscription_id = 'YOUR_SUBSCRIPTION_ID'
resource_group = 'COMPSCI532'
workspace_name = 'COMPSCI532-ML-Workspace'

workspace = Workspace(subscription_id, resource_group, workspace_name)

dataset = Dataset.get_by_name(workspace, name='Get-Exit-Lists')
dataset.to_pandas_dataframe()

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.1.1) and mlflow-skinny (2.22.1) are different. This may lead to unexpected behavior. Please install the same version of both packages.
  mlflow.mismatch._check_version_mismatch()


,source,type,timestamp,EventProcessedUtcTime,PartitionId,EventEnqueuedUtcTime,fingerprint,published,last_status,exit_ip,exit_seen
0,Get_Exit_Lists,exit_list,2025-11-27T01:30:08.411203+00:00,None,8,None,746FDA7D7F633FF88BF735416A42E472D630D868,2025-11-26 14:19:30,2025-11-26 17:00:00,178.20.55.182,2025-11-26 17:34:03
1,Get_Exit_Lists,exit_list,2025-11-27T01:30:08.513559+00:00,None,8,None,D25E49EF7FD775352084AC86AF030F8DE9036359,2025-11-26 04:55:08,2025-11-26 17:00:00,45.141.215.28,2025-11-26 17:21:31
2,Get_Exit_Lists,exit_list,2025-11-27T01:30:08.409622+00:00,None,8,None,7D2EEBF22F0E91025E8E9BF739D8B8237577448B,2025-11-26 07:22:11,2025-11-26 17:00:00,193.189.100.195,2025-11-26 17:25:15
3,Get_Exit_Lists,exit_list,2025-11-27T01:30:08.408749+00:00,None,8,None,98AE10E67739CCC9FAD8B223236BBB080C3B0852,2025-11-26 05:24:20,2025-11-26 17:00:00,45.95.169.104,2025-11-26 17:12:56
4,Get_Exit_Lists,exit_list,2025-11-27T01:30:08.610261+00:00,None,8,None,5D98A8A2F60F26C65E34F4205BE77219E10EFB09,2025-11-26 06:47:17,2025-11-26 18:00:00,45.141.215.19,2025-11-26 18:35:13
...,...,...,...,...,...,...,...,...,...,...,...
3249,Get_Exit_Lists,exit_list,2025-11-27T01:30:08.377451+00:00,None,8,None,D6DC1CD60C83FE9252D49292D49568E917B1C260,2025-11-26 11:31:50,2025-11-26 15:00:00,192.42.116.198,2025-11-26 15:13:46
3250,Get_Exit_Lists,exit_list,2025-11-27T01:30:08.550537+00:00,None,8,None,99D65135D343EB8549B2D46C4EF8CA71C6C91ADD,2025-11-26 12:32:49,2025-11-26 18:00:00,192.42.116.197,2025-11-26 18:08:00
3251,Get_Exit_Lists,exit_list,2025-11-27T01:30:08.488314+00:00,None,8,None,34CA0E8F7838FCC9984C7BAA282884A4F843A423,2025-11-26 01:34:45,2025-11-26 17:00:00,45.138.16.107,2025-11-26 17:19:17
3252,Get_Exit_Lists,exit_list,2025-11-27T01:30:08.363742+00:00,None,8,None,D50B00B8199F76FA6D1FB448AAC1BA9EEB4CD400,2025-11-26 07:14:16,2025-11-26 17:00:00,67.219.109.141,2025-11-26 17:44:20


In [8]:
%pip install ipwhois azure-storage-file-datalake pyarrow


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import time
import ipaddress
import requests
from ipwhois import IPWhois
from concurrent.futures import ThreadPoolExecutor

# --- 1. Load dataset and filter valid public IPs ---
df = dataset.to_pandas_dataframe()

def is_public_ip(value):
    try:
        ip = ipaddress.ip_address(value)
        return ip.is_global
    except ValueError:
        return False

df = df[df["exit_ip"].apply(is_public_ip)]
base_columns = ["fingerprint", "published", "last_status", "exit_ip", "exit_seen"]
df_subset = df[base_columns].copy()

# --- 2. API Keys ---
VT_API_KEY = "YOUR_VT_API_KEY"
CENSYS_API_ID = "YOUR_CENSYS_API_ID"
CENSYS_API_SECRET = "YOUR_CENSYS_API_SECRET"
OTX_API_KEY = "YOUR_OTX_API_KEY"
ABUSEIPDB_API_KEY = "YOUR_ABUSEIPDB_API_KEY"
NVD_API_KEY = "YOUR_NVD_API_KEY"  

# --- 3. Provider functions ---

def vt_ip_lookup(ip):
    url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"
    headers = {"x-apikey": VT_API_KEY}
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        return {"error": str(e)}

def censys_bulk_lookup(ip_list):
    url = "https://api.platform.censys.io/v3/global/asset/host/list"
    headers = {
        "Authorization": f"Bearer {CENSYS_API_SECRET}",
        "Accept": "application/json"
    }
    payload = {"hosts": ip_list}
    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=30)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        return {"error": str(e)}

def otx_ip_lookup(ip):
    url = f"https://otx.alienvault.com/api/v1/indicators/IPv4/{ip}/general"
    headers = {"X-OTX-API-KEY": OTX_API_KEY}
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        return {"error": str(e)}

def otx_cve_lookup(ip):
    url = f"https://otx.alienvault.com/api/v1/indicators/IPv4/{ip}/cves"
    headers = {"X-OTX-API-KEY": OTX_API_KEY}
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()
        return resp.json().get("cves", [])
    except Exception as e:
        return []

def abuseipdb_ip_lookup(ip):
    url = "https://api.abuseipdb.com/api/v2/check"
    headers = {
        "Key": ABUSEIPDB_API_KEY,
        "Accept": "application/json"
    }
    params = {"ipAddress": ip, "maxAgeInDays": 30}
    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        return {"error": str(e)}

def nvd_cve_lookup(cve_id, api_key=NVD_API_KEY):
    """
    Look up CVE details from NVD API and return CVSS v3 metadata only.
    
    Parameters:
        cve_id (str): CVE identifier (e.g., "CVE-2021-34527")
        api_key (str): NVD API key (defaults to your NVD_API_KEY variable)
    
    Returns:
        dict: Metadata including CVSS v3 score, severity, and dates
    """
    url = f"https://services.nvd.nist.gov/rest/json/cve/1.0/{cve_id}"
    headers = {"apiKey": api_key} if api_key else {}

    try:
        r = requests.get(url, headers=headers, timeout=15)
        r.raise_for_status()
        data = r.json()

        items = data.get("result", {}).get("CVE_Items", [])
        if not items:
            return {
                "cve_id": cve_id,
                "cvss_score": None,
                "cvss_severity": None,
                "published_date": None,
                "last_modified_date": None,
                "nvd_url": url
            }

        item = items[0]
        published = item.get("publishedDate")
        modified = item.get("lastModifiedDate")

        # CVSS v3 only
        metrics_v3 = item.get("impact", {}).get("baseMetricV3", {})
        cvss_v3 = metrics_v3.get("cvssV3", {})
        cvss_score = cvss_v3.get("baseScore")
        cvss_severity = cvss_v3.get("baseSeverity")

        return {
            "cve_id": cve_id,
            "cvss_score": cvss_score,
            "cvss_severity": cvss_severity,
            "published_date": published,
            "last_modified_date": modified,
            "nvd_url": url
        }

    except Exception as e:
        return {
            "cve_id": cve_id,
            "cvss_score": None,
            "cvss_severity": None,
            "published_date": None,
            "last_modified_date": None,
            "nvd_url": url,
            "error": str(e)
        }

# --- 4. WHOIS enrichment ---
def safe_whois(ip):
    try:
        obj = IPWhois(ip)
        rdap = obj.lookup_rdap(depth=1)  # depth=1 gives you entities too
        return rdap
    except Exception as e:
        return {"error": str(e)}


# --- 5. Load CISA KEV dataset ---
kev_df = pd.read_csv(
    "https://raw.githubusercontent.com/cisagov/kev-data/main/known_exploited_vulnerabilities.csv"
)
kev_df["cveID"] = kev_df["cveID"].astype(str).str.strip().str.upper()
kev_cves = set(kev_df["cveID"].dropna().unique())

# --- 6. Enrichment function with caching ---
cache = {}

def enrich_ip(ip, censys_results_map):
    results = {
        "exit_ip": ip,
        "virustotal": vt_ip_lookup(ip),
        "censys": censys_results_map.get(ip, {}),
        "otx": otx_ip_lookup(ip),
        "otx_cves": otx_cve_lookup(ip),
        "abuseipdb": abuseipdb_ip_lookup(ip),
        "whois_asn": safe_whois(ip),
    }

# --- Extract CVEs from both pulse_info and direct CVE lookup ---
    pulse_cves = []
    pulses = results["otx"].get("pulse_info", {}).get("pulses", [])
    for pulse in pulses:
        if "cve" in pulse:
            pulse_cves.extend(pulse["cve"])

    # Normalize and merge CVEs
    norm = lambda s: s.strip().upper()
    all_cves = set(norm(c) for c in pulse_cves + results["otx_cves"] if isinstance(c, str))
    results["cves"] = sorted(all_cves)

    # Extract CVEs, file hashes, URLs, domains from OTX pulses
    cves, file_hashes, urls, domains = [], [], [], []
    pulses = results["otx"].get("pulse_info", {}).get("pulses", [])
    for pulse in pulses:
        if "cve" in pulse:
            cves.extend(pulse["cve"])
        for ind in pulse.get("indicators", []):
            t, v = ind.get("type"), ind.get("indicator")
            if t in ["FileHash-MD5", "FileHash-SHA1", "FileHash-SHA256"]:
                file_hashes.append(v)
            elif t == "URL":
                urls.append(v)
            elif t == "domain":
                domains.append(v)

    # Normalize CVEs
    norm = lambda s: s.strip().upper()
    results["cves"] = sorted({norm(c) for c in cves if isinstance(c, str)})
    results["file_hashes"] = list(set(file_hashes))
    results["urls"] = list(set(urls))
    results["domains"] = list(set(domains))

    # KEV flags
    kev_flags = {cve: 1 if cve in kev_cves else 0 for cve in results["cves"]}
    results["IsCISAKEV"] = kev_flags

    cache[ip] = results
    return results

# --- 7. Run enrichment in parallel ---
ips = df_subset["exit_ip"].tolist()

# Chunk IPs for Censys bulk lookup
chunk_size = 500
censys_results_map = {}
for i in range(0, len(ips), chunk_size):
    chunk = ips[i:i+chunk_size]
    censys_resp = censys_bulk_lookup(chunk)
    if "result" in censys_resp:
        for host in censys_resp["result"]:
            censys_results_map[host["ip"]] = host

with ThreadPoolExecutor(max_workers=20) as executor:
    results = list(executor.map(lambda ip: enrich_ip(ip, censys_results_map), ips))

df_subset["provider_results"] = results

# --- 8. Collect all CVEs across IPs for EPSS lookup ---
all_cves = sorted({cve for r in results for cve in r.get("cves", [])})
epss_map = {}

def fetch_epss_for_cve(cve):
    try:
        resp = requests.get(
            "https://api.first.org/data/v1/epss",
            params={"cve": cve},
            timeout=15
        )
        resp.raise_for_status()
        data = resp.json().get("data", [])
        if data:
            entry = data[0]
            epss_map[cve] = {
                "epss": entry.get("epss"),
                "percentile": entry.get("percentile")
            }
    except Exception as e:
        epss_map[cve] = {"epss": None, "percentile": None, "error": str(e)}

# Run EPSS lookups in parallel
with ThreadPoolExecutor(max_workers=10) as executor:
    executor.map(fetch_epss_for_cve, all_cves)

# Attach EPSS results back into provider_results
for r in results:
    r["EPSS"] = {cve: epss_map.get(cve, {}) for cve in r.get("cves", [])}

# --- 9. Extended Flattening ---
def flatten_provider_data(r):
    vt_attrs = r.get("virustotal", {}).get("data", {}).get("attributes", {})
    censys_result = r.get("censys", {}).get("result", {})
    otx_result = r.get("otx", {})
    abuse_result = r.get("abuseipdb", {}).get("data", {})
    whois_data = r.get("whois_asn", {})

    pulses = otx_result.get("pulse_info", {}).get("pulses", [])
    pulse_names = [p.get("name") for p in pulses]
    pulse_authors = [p.get("author_name") for p in pulses]

    return pd.Series({
        # VirusTotal
        "VT_Reputation": vt_attrs.get("reputation"),
        "VT_LastAnalysisStats": vt_attrs.get("last_analysis_stats"),
        "VT_LastAnalysisResults": vt_attrs.get("last_analysis_results"),
        "VT_Country": vt_attrs.get("country"),
        "VT_Network": vt_attrs.get("network"),
        "VT_ASOwner": vt_attrs.get("as_owner"),
        "VT_WHOIS": vt_attrs.get("whois"),

        # Censys
        "Services": censys_result.get("services", []),
        "Location": censys_result.get("location", {}),
        "ASN": censys_result.get("autonomous_system", {}).get("asn"),
        "ASOrg": censys_result.get("autonomous_system", {}).get("organization"),
        "DNS": censys_result.get("dns", {}),
        "Tags": censys_result.get("tags", []),

        # OTX
        "OTX_Indicator": otx_result.get("indicator"),
        "OTX_Related": otx_result.get("related"),
        "OTX_PulseNames": ", ".join([n for n in pulse_names if n]),
        "OTX_PulseAuthors": ", ".join([a for a in pulse_authors if a]),
        "OTX_CVE_List": r.get("cves", []),
        "OTX_FileHashes": ", ".join(r.get("file_hashes", [])),
        "OTX_URLs": ", ".join(r.get("urls", [])),
        "OTX_Domains": ", ".join(r.get("domains", [])),

        # AbuseIPDB
        "AbuseIPDB_TotalReports": abuse_result.get("totalReports"),
        "AbuseIPDB_NumDistinctUsers": abuse_result.get("numDistinctUsers"),
        "AbuseIPDB_LastReportedAt": abuse_result.get("lastReportedAt"),
        "AbuseIPDB_IsWhitelisted": abuse_result.get("isWhitelisted"),
        "AbuseIPDB_CountryCode": abuse_result.get("countryCode"),
        "AbuseIPDB_UsageType": abuse_result.get("usageType"),
        "AbuseIPDB_Score": abuse_result.get("abuseConfidenceScore"),

        # IPWhois
        "WHOIS_ASN": whois_data.get("asn"),
        "WHOIS_ASN_Description": whois_data.get("asn_description"),
        "WHOIS_ASN_CountryCode": whois_data.get("asn_country_code"),
        "WHOIS_ASN_Registry": whois_data.get("asn_registry"),
        "WHOIS_Network": whois_data.get("network"),
        "WHOIS_Entities": whois_data.get("entities"),

        # Summary
        "IsCISAKEV": 1 if any(v == 1 for v in r.get("IsCISAKEV", {}).values()) else 0,
        "MaxEPSS": max([float(v["epss"]) for v in r.get("EPSS", {}).values()], default=0),
        "CVE_List": r.get("cves", []),
        "LookupTimeSec": r.get("lookup_time_sec")
    })

flat_df = df_subset["provider_results"].apply(flatten_provider_data)
df_final = pd.concat([df_subset, flat_df], axis=1)

# --- 10. CVE-level dataframe ---
df_cve = df_final.explode("CVE_List").dropna(subset=["CVE_List"])

if not df_cve.empty:
    # Attach EPSS and KEV only for CVEs present
    df_cve = df_cve.assign(
        EPSS_Score=[
            float(r.get("EPSS", {}).get(cve, {}).get("epss", 0))
            for r, cve in zip(df_cve["provider_results"], df_cve["CVE_List"])
        ],
        EPSS_Percentile=[
            float(r.get("EPSS", {}).get(cve, {}).get("percentile", 0))
            for r, cve in zip(df_cve["provider_results"], df_cve["CVE_List"])
        ],
        IsCISAKEV_CVE=[
            int(r.get("IsCISAKEV", {}).get(cve, 0))
            for r, cve in zip(df_cve["provider_results"], df_cve["CVE_List"])
        ]
    )

    # --- 10b. NVD CVSS enrichment ---
    unique_cves = df_cve["CVE_List"].dropna().unique()
    cve_metadata = [nvd_cve_lookup(cve, NVD_API_KEY) for cve in unique_cves]
    cve_df = pd.DataFrame(cve_metadata)

    if not cve_df.empty and "cve_id" in cve_df.columns:
        df_cve = df_cve.merge(cve_df, left_on="CVE_List", right_on="cve_id", how="left")
    else:
        df_cve["cvss_score"] = None
        df_cve["cvss_severity"] = None
        df_cve["published_date"] = None
        df_cve["last_modified_date"] = None

    # --- Aggregate back to IP level ---
    agg_cve = (
        df_cve.groupby("exit_ip")
        .agg({
            "CVE_List": lambda x: list(set(x.dropna())),
            "EPSS_Score": list,
            "EPSS_Percentile": list,
            "IsCISAKEV_CVE": list,
            "cvss_score": list,
            "cvss_severity": list,
            "published_date": list,
            "last_modified_date": list,
        })
        .reset_index()
    )

    # Merge into df_final
    df_final = df_final.merge(agg_cve, on="exit_ip", how="left")

# Now safe to drop provider_results
df_final = df_final.drop(columns=["provider_results", "WHOIS_Network", "Location", "DNS"], errors="ignore")

# --- 11. Display results ---
print("=== IP-Level Enrichment ===")
display(df_final.head(20))

print("=== CVE-Level Enrichment ===")
display(df_cve.head(20))

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
=== IP-Level Enrichment ===


,fingerprint,published,last_status,exit_ip,exit_seen,VT_Reputation,VT_LastAnalysisStats,VT_LastAnalysisResults,VT_Country,VT_Network,...,AbuseIPDB_Score,WHOIS_ASN,WHOIS_ASN_Description,WHOIS_ASN_CountryCode,WHOIS_ASN_Registry,WHOIS_Entities,IsCISAKEV,MaxEPSS,CVE_List,LookupTimeSec
0,746FDA7D7F633FF88BF735416A42E472D630D868,2025-11-26 14:19:30,2025-11-26 17:00:00,178.20.55.182,2025-11-26 17:34:03,None,None,None,None,None,...,None,29075,"IELO IELO Main Network, FR",FR,ripencc,"[MNT-LIAZO, NOAC1-RIPE, NOTC1-RIPE, ORG-NO16-R...",0,0,[],None
1,D25E49EF7FD775352084AC86AF030F8DE9036359,2025-11-26 04:55:08,2025-11-26 17:00:00,45.141.215.28,2025-11-26 17:21:31,None,None,None,None,None,...,None,210558,"SERVICES-1337-GMBH 1337-SERVICES-GMBH-NETWORK, DE",NL,ripencc,"[ORG-SG415-RIPE, PREFIXBROKER-MNT, SGAH7-RIPE,...",0,0,[],None
2,7D2EEBF22F0E91025E8E9BF739D8B8237577448B,2025-11-26 07:22:11,2025-11-26 17:00:00,193.189.100.195,2025-11-26 17:25:15,None,None,None,None,None,...,None,41281,"KEFF Interplanetary Communications Network, GB",GB,ripencc,"[KEFF1-RIPE, MNT-KEFF, ORG-KNL18-RIPE, RIPE-NC...",0,0,[],None
3,98AE10E67739CCC9FAD8B223236BBB080C3B0852,2025-11-26 05:24:20,2025-11-26 17:00:00,45.95.169.104,2025-11-26 17:12:56,None,None,None,None,None,...,None,211619,"MAXKO, HR",HR,ripencc,"[DF8797-RIPE, mnt-hr-maxko-1, ORG-MJ181-RIPE, ...",0,0,[],None
4,5D98A8A2F60F26C65E34F4205BE77219E10EFB09,2025-11-26 06:47:17,2025-11-26 18:00:00,45.141.215.19,2025-11-26 18:35:13,None,None,None,None,None,...,None,210558,"SERVICES-1337-GMBH 1337-SERVICES-GMBH-NETWORK, DE",NL,ripencc,"[ORG-SG415-RIPE, PREFIXBROKER-MNT, SGAH7-RIPE,...",0,0,[],None
5,5E42C344C22A94C518A2DB764F66998CA52349F0,2025-11-26 11:20:40,2025-11-26 18:00:00,209.141.45.141,2025-11-26 18:16:14,None,None,None,None,None,...,None,53667,"PONYNET, US",US,arin,[SYNDI-5],0,0,[],None
6,4BFC9C631A93FF4BA3AA84BC6931B4310C38A263,2025-11-26 12:33:59,2025-11-26 18:00:00,109.70.100.4,2025-11-26 18:45:57,None,None,None,None,None,...,None,208323,"APPLIEDPRIVACY-AS, AT",AT,ripencc,"[APPLIEDPRIVACY-MNT, FFAP1-RIPE, ORG-PRIV3-RIP...",0,0,[],None
7,81F4867EC51E06053346C0226FB82AC8D14BE4D2,2025-11-26 10:15:26,2025-11-26 17:00:00,171.25.193.235,2025-11-26 17:35:53,None,None,None,None,None,...,None,198093,DFRI-AS Foreningen for digitala fri- och ratti...,SE,ripencc,"[DFRI-MNT, EJ1830-RIPE, ER6905-RIPE, JN9999, O...",0,0,[],None
8,A9EB576362462F722CD3EB67961E768E41228F2D,2025-11-26 06:59:14,2025-11-26 17:00:00,124.198.131.108,2025-11-26 17:03:41,None,None,None,None,None,...,None,210558,"SERVICES-1337-GMBH 1337-SERVICES-GMBH-NETWORK, DE",NL,ripencc,"[ORG-SG468-RIPE, PREFIXBROKER-MNT, SGAH18-RIPE...",0,0,[],None
9,DF4E4ADB8035B3AEB6B426A6A44DDE691810EB51,2025-11-26 13:20:28,2025-11-26 17:00:00,45.138.16.231,2025-11-26 17:09:24,None,None,None,None,None,...,None,210558,"SERVICES-1337-GMBH 1337-SERVICES-GMBH-NETWORK, DE",NL,ripencc,"[ORG-SG413-RIPE, PREFIXBROKER-MNT, SGAH5-RIPE,...",0,0,[],None


=== CVE-Level Enrichment ===


,fingerprint,published,last_status,exit_ip,exit_seen,provider_results,VT_Reputation,VT_LastAnalysisStats,VT_LastAnalysisResults,VT_Country,...,WHOIS_ASN,WHOIS_ASN_Description,WHOIS_ASN_CountryCode,WHOIS_ASN_Registry,WHOIS_Network,WHOIS_Entities,IsCISAKEV,MaxEPSS,CVE_List,LookupTimeSec


In [ ]:
import io
import pyarrow as pa
import pyarrow.parquet as pq
from azure.storage.filedatalake import DataLakeServiceClient

# Convert dataframe to Arrow table
table = pa.Table.from_pandas(df_final)

# Serialize to Parquet in memory
buffer = io.BytesIO()
pq.write_table(table, buffer)
buffer.seek(0)

# Connect to ADLS Gen2
account_name = "YOUR_ACCOUNT_NAME"
account_key = "YOUR_ACCOUNT_KEY"
container_name = "get-exit-lists-curated"
directory_name = "exports"
file_name = "ip_enrichment_2.parquet"

service_client = DataLakeServiceClient(
    account_url=f"https://{account_name}.dfs.core.windows.net",
    credential=account_key
)

# Get filesystem (container)
file_system_client = service_client.get_file_system_client(file_system=container_name)

# Get or create directory
directory_client = file_system_client.get_directory_client(directory_name)
try:
    directory_client.create_directory()
except Exception:
    pass  # already exists

# Create file and upload parquet bytes
file_client = directory_client.create_file(file_name)
file_client.append_data(data=buffer.getvalue(), offset=0, length=buffer.getbuffer().nbytes)
file_client.flush_data(buffer.getbuffer().nbytes)

{'date': datetime.datetime(2025, 12, 5, 0, 59, 20, tzinfo=datetime.timezone.utc),
 'etag': '"0x8DE339986A1A619"',
 'last_modified': datetime.datetime(2025, 12, 5, 0, 59, 21, tzinfo=datetime.timezone.utc),
 'content_length': 0,
 'client_request_id': 'a26b78d6-d175-11f0-aff0-6045bd857210',
 'request_id': '941cde47-e01f-00c2-5c82-65cd7d000000',
 'version': '2025-05-05',
 'request_server_encrypted': False,
 'encryption_key_sha256': None,
 'lease_renewed': None}

In [36]:

import pandas as pd
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
data_asset = ml_client.data.get("get-exit-lists-ti-enriched", version="1")

df = pd.read_parquet(data_asset.path)
df = df.dropna(axis=1, how="all")
df = df.drop(columns=["Services", "Tags", "OTX_PulseAuthors", "OTX_CVE_List", "OTX_FileHashes", "OTX_URLs", "OTX_Domains", "IsCISAKEV", "MaxEPSS", "CVE_List", "VT_LastAnalysisStats", ])
# Drop rows where all values are NaN
df = df.dropna(how="any")
df

Found the config file in: /config.json


,fingerprint,published,last_status,exit_ip,exit_seen,VT_Reputation,VT_LastAnalysisStats,VT_LastAnalysisResults,VT_Country,VT_Network,...,AbuseIPDB_LastReportedAt,AbuseIPDB_IsWhitelisted,AbuseIPDB_CountryCode,AbuseIPDB_UsageType,AbuseIPDB_Score,WHOIS_ASN,WHOIS_ASN_Description,WHOIS_ASN_CountryCode,WHOIS_ASN_Registry,WHOIS_Entities
0,746FDA7D7F633FF88BF735416A42E472D630D868,2025-11-26 14:19:30,2025-11-26 17:00:00,178.20.55.182,2025-11-26 17:34:03,-5.0,"{'harmless': 54.0, 'malicious': 9.0, 'suspicio...","{'0xSI_f33d': {'category': 'undetected', 'engi...",FR,178.20.48.0/21,...,2025-12-05T00:05:03+00:00,False,FR,Data Center/Web Hosting/Transit,100.0,29075,"IELO IELO Main Network, FR",FR,ripencc,"[MNT-LIAZO, NOAC1-RIPE, NOTC1-RIPE, ORG-NO16-R..."
1,D25E49EF7FD775352084AC86AF030F8DE9036359,2025-11-26 04:55:08,2025-11-26 17:00:00,45.141.215.28,2025-11-26 17:21:31,-3.0,"{'harmless': 55.0, 'malicious': 9.0, 'suspicio...","{'0xSI_f33d': {'category': 'undetected', 'engi...",PL,45.141.215.0/24,...,2025-12-04T21:36:41+00:00,False,PL,Data Center/Web Hosting/Transit,97.0,210558,"SERVICES-1337-GMBH 1337-SERVICES-GMBH-NETWORK, DE",NL,ripencc,"[ORG-SG415-RIPE, PREFIXBROKER-MNT, SGAH7-RIPE,..."
2,7D2EEBF22F0E91025E8E9BF739D8B8237577448B,2025-11-26 07:22:11,2025-11-26 17:00:00,193.189.100.195,2025-11-26 17:25:15,-26.0,"{'harmless': 52.0, 'malicious': 13.0, 'suspici...","{'0xSI_f33d': {'category': 'undetected', 'engi...",SE,193.189.100.0/24,...,2025-12-04T21:36:46+00:00,False,SE,Data Center/Web Hosting/Transit,98.0,41281,"KEFF Interplanetary Communications Network, GB",GB,ripencc,"[KEFF1-RIPE, MNT-KEFF, ORG-KNL18-RIPE, RIPE-NC..."
3,98AE10E67739CCC9FAD8B223236BBB080C3B0852,2025-11-26 05:24:20,2025-11-26 17:00:00,45.95.169.104,2025-11-26 17:12:56,-13.0,"{'harmless': 52.0, 'malicious': 13.0, 'suspici...","{'0xSI_f33d': {'category': 'undetected', 'engi...",HR,45.95.168.0/22,...,2025-12-04T21:36:42+00:00,False,HR,Data Center/Web Hosting/Transit,64.0,211619,"MAXKO, HR",HR,ripencc,"[DF8797-RIPE, mnt-hr-maxko-1, ORG-MJ181-RIPE, ..."
5,5E42C344C22A94C518A2DB764F66998CA52349F0,2025-11-26 11:20:40,2025-11-26 18:00:00,209.141.45.141,2025-11-26 18:16:14,-2.0,"{'harmless': 61.0, 'malicious': 2.0, 'suspicio...","{'0xSI_f33d': {'category': 'undetected', 'engi...",US,209.141.32.0/19,...,2025-12-04T15:05:16+00:00,False,US,Data Center/Web Hosting/Transit,39.0,53667,"PONYNET, US",US,arin,[SYNDI-5]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
708,72F3AC8E95AD980DC5F0FCF29FDB1CE72128729E,2025-11-26 07:53:01,2025-11-26 18:00:00,109.70.100.66,2025-11-26 18:38:46,-4.0,"{'harmless': 52.0, 'malicious': 11.0, 'suspici...","{'0xSI_f33d': {'category': 'undetected', 'engi...",AT,109.70.100.0/24,...,2025-12-04T23:31:49+00:00,False,AT,University/College/School,100.0,208323,"APPLIEDPRIVACY-AS, AT",AT,ripencc,"[APPLIEDPRIVACY-MNT, FFAP1-RIPE, ORG-PRIV3-RIP..."
709,5D9D80195162D7D77506EAF768F00F70A51CD191,2025-11-26 14:02:01,2025-11-26 17:00:00,203.55.81.1,2025-11-26 17:11:37,-3.0,"{'harmless': 54.0, 'malicious': 9.0, 'suspicio...","{'0xSI_f33d': {'category': 'undetected', 'engi...",FR,203.55.81.0/24,...,2025-12-04T21:46:32+00:00,False,FR,Data Center/Web Hosting/Transit,100.0,213873,"OUIDO-HOSTING, FR",FR,ripencc,"[lir-fr-julesd-1-MNT, NOC293-RIPE, ORG-MS374-R..."
710,0AF0BA36BB1D55C8C66C2441F96286F43ADEA164,2025-11-26 07:05:06,2025-11-26 17:00:00,45.84.107.47,2025-11-26 17:24:55,-4.0,"{'harmless': 54.0, 'malicious': 8.0, 'suspicio...","{'0xSI_f33d': {'category': 'undetected', 'engi...",SE,45.84.107.0/24,...,2025-12-05T00:04:02+00:00,False,SE,Data Center/Web Hosting/Transit,100.0,214503,"R0CKET-CLOUD, SE",SE,ripencc,"[MNT-QUXLABS, NA8786-RIPE, AR75103-RIPE]"
711,ECE831534426279E32E530CDC9279D36E2A6AC60,2025-11-26 16:10:35,2025-11-26 18:00:00,5.255.97.221,2025-11-26 18:09:38,-1.0,"{'harmless': 58.0, 'malicious': 4.0, 'suspicio...","{'0xSI_f33d': {'category': 'undetected', 'engi...",NL,5.255.96.0/19,...,2025-11-28T18:38:13+00:00,False,NL,Data

In [ ]:
import io
import pyarrow as pa
import pyarrow.parquet as pq
from azure.storage.filedatalake import DataLakeServiceClient

# Convert dataframe to Arrow table
table = pa.Table.from_pandas(df)

# Serialize to Parquet in memory
buffer = io.BytesIO()
pq.write_table(table, buffer)
buffer.seek(0)

# Connect to ADLS Gen2
account_name = "YOUR_ACCOUNT_NAME"
account_key = "YOUR_ACCOUNT_KEY"
container_name = "get-exit-lists-curated"
directory_name = "exports"
file_name = "ip_enrichment_4.parquet"

service_client = DataLakeServiceClient(
    account_url=f"https://{account_name}.dfs.core.windows.net",
    credential=account_key
)

# Get filesystem (container)
file_system_client = service_client.get_file_system_client(file_system=container_name)

# Get or create directory
directory_client = file_system_client.get_directory_client(directory_name)
try:
    directory_client.create_directory()
except Exception:
    pass  # already exists

# Create file and upload parquet bytes
file_client = directory_client.create_file(file_name)
file_client.append_data(data=buffer.getvalue(), offset=0, length=buffer.getbuffer().nbytes)
file_client.flush_data(buffer.getbuffer().nbytes)

{'date': datetime.datetime(2025, 12, 5, 2, 13, 29, tzinfo=datetime.timezone.utc),
 'etag': '"0x8DE33A3E22EEA2D"',
 'last_modified': datetime.datetime(2025, 12, 5, 2, 13, 30, tzinfo=datetime.timezone.utc),
 'content_length': 0,
 'client_request_id': 'fdf93f58-d17f-11f0-aff0-6045bd857210',
 'request_id': '41999a4e-e01f-00d2-3c8c-650815000000',
 'version': '2025-05-05',
 'request_server_encrypted': False,
 'encryption_key_sha256': None,
 'lease_renewed': None}